# Configure training per client

Different sites may need different batch sizes, local epochs, or learning rates. Keep the training code shared and pass site-specific arguments through `set_per_site_config()`.

Complete [setup](../01.1_running_federated_learning_job/setup.ipynb). This lesson configures client-side training through the same Recipe API used for the server.

In [ ]:
import os
import sys
from pathlib import Path
from IPython.display import Code, display

# Make notebook shell commands use the selected kernel's environment.
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", "")

lesson = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "customize_client_training.ipynb").is_file())
code = lesson / "code"
assert (code / "job.py").is_file(), f"Keep this notebook beside its code directory: {code}"
print("Example code:", code)

In [ ]:
display(Code(filename=str(code / "job.py"), language="python"))

Call `set_per_site_config()` immediately after constructing the recipe, before other client customizations. The keys must match the participating site names. Because these clients are explicitly named, use `SimEnv(clients=["site-1", "site-2"], log_config="progress")` to start those sites.

| Site | Learning rate | Batch size | Samples / epoch | Optimizer steps / round |
| --- | ---: | ---: | ---: | ---: |
| site-1 | 0.1 | 32 | 200 | 7 |
| site-2 | 0.05 | 16 | 200 | 13 |

`client.py` parses these arguments and prints its effective settings. Each site still generates its own samples. Configuration values travel with the exported job; use site-local mechanisms for credentials rather than putting secrets in training arguments.

Run the next cell to execute this section's `job.py`.

Compare the sites' final evaluation scores in the run summary. To see each site's effective batch size and learning rate in the console, add `--log_config full` to the run command.

In [ ]:
%cd {code}
!python job.py
assert _exit_code == 0, "Command failed; see the output above."

## Training choices affect aggregation

This FedAvg configuration weights contributions by optimizer-step count. Equal dataset sizes do **not** imply equal weights when batch sizes or local epochs differ: site-2 performs more steps here. Report actual work honestly; if your algorithm needs example-count weighting, configure the aggregation policy deliberately instead of mislabeling a sample count as a step count.

**Try it:** give both sites batch size 32, run in a fresh workspace, and compare their reported steps. Then change only site-2's local epochs.

**Check yourself:** Which settings belong to the shared recipe and which belong to individual sites? What would happen if the per-site key were `site-3`, but the simulation started only `site-1` and `site-2`?

Next: [experiment tracking](../01.5_experiment_tracking/experiment_tracking.ipynb).